# 12 - IHDP semi-synthetic benchmark

Notebook 10 applied every estimator here to the Lalonde data and none recovered
the experimental benchmark. That is one kind of failure. This notebook shows a
different one, which is harder to notice and in some ways more instructive.

IHDP takes covariates and treatment assignment from a real trial — the Infant
Health and Development Program — and simulates the outcomes. The confounding
structure is real; the true individual effects are known exactly. So estimators
can be scored, not merely compared with each other.

The result, stated up front because the notebook is built around it: **the
unadjusted difference in means is the most accurate estimator on this
benchmark**, and it beats doubly robust estimation in 8 of 10 replications.

## Causal question

What was the average effect of the intervention on the children who received
it, and how much does that effect vary between children?

## Data and design

- **Unit of analysis:** one child. 747 per replication — 139 treated, 608
  control.
- **Treatment:** `treatment`, from the real programme. A non-random subset of
  treated children was removed to induce imbalance, which is what makes the
  benchmark a test of adjustment.
- **Outcome:** `outcome`, the factual simulated outcome — the only one an
  analyst would observe.
- **Covariates:** `x1`–`x25`, real measurements on the children and mothers.
- **Ground truth:** `true_ite`, and the surfaces `mu0` / `mu1` it comes from.

**Ten replications.** Each redraws the simulated outcomes from the same
covariates and the same treatment assignment. That matters more than it sounds:
an estimator's error on one replication is a single draw, and this benchmark is
usually reported across many for exactly that reason.

Run `python scripts/prepare_ihdp_dataset.py` first if the file is missing.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor

from causal_inference_lab.diagnostics import balance_table, ipw_weights
from causal_inference_lab.dml import double_machine_learning_ate
from causal_inference_lab.estimators import aipw_ate, difference_in_means, g_computation_ate, ipw_ate
from causal_inference_lab.meta_learners import SMetaLearner, TMetaLearner
from causal_inference_lab.uncertainty import bootstrap_ate

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ihdp_benchmark.csv"
if not DATA_PATH.exists():
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
    from prepare_ihdp_dataset import prepare_ihdp_dataset

    prepare_ihdp_dataset()

data = pd.read_csv(DATA_PATH)
COVARIATES = [f"x{index}" for index in range(1, 26)]
first = data.loc[data["replication"] == 1]

print(f"rows:           {len(data):,} across {data['replication'].nunique()} replications")
print(f"per replication:{len(first)}  ({int(first['treatment'].sum())} treated, "
      f"{int(len(first) - first['treatment'].sum())} control)")
print()
print("true effect and outcome scale by replication:")
print(
    data.groupby("replication")
    .agg(true_ate=("true_ite", "mean"), ite_sd=("true_ite", "std"), outcome_sd=("outcome", "std"))
    .to_string(float_format=lambda v: f"{v:.2f}")
)

rows:           7,470 across 10 replications
per replication:747  (139 treated, 608 control)

true effect and outcome scale by replication:
             true_ate  ite_sd  outcome_sd
replication                              
1                4.02    0.86        2.18
2                4.05    0.82        2.09
3                4.10    0.92        2.28
4                4.27    1.94        2.79
5                4.16    2.62        3.30
6                4.00    0.79        2.21
7                3.99    0.29        1.90
8                3.85    1.54        2.60
9               10.47   27.38       25.17
10               4.59    8.89        8.67


**Interpretation.** Nine of the ten replications have a true ATE near 4 with a
modest outcome spread. Replication 9 does not: its true effect is 10.5, its
individual effects have a standard deviation of 27, and its outcomes run out to
255 against a maximum near 11 elsewhere.

That is a property of the benchmark, not a data error — the simulated response
surfaces include a multiplicative regime that produces heavy tails. It matters
because averaging any error metric across these ten replications means averaging
across two quite different problems, and one of them will dominate.

## Estimand

Two, and IHDP is used for both.

The **ATE** — the average effect across children — scored by absolute error
against the known truth.

The **CATE**, scored by **PEHE**: the root mean squared error of the estimated
individual effects against `true_ite`. This is the metric IHDP is best known for
in the literature, and it asks a strictly harder question than the ATE, because
it is graded per child rather than on average.

## Identification assumptions

1. **Conditional ignorability** given `x1`–`x25`. Unusually for a real dataset,
   this one is true by construction: the outcomes were simulated from these
   covariates, so nothing else confounds. That is the benchmark's whole point —
   any error we see is estimation error, not unmeasured confounding.
2. **Overlap.** Not guaranteed. Treated units were removed non-randomly, so some
   regions of covariate space have few treated children. Checked below.
3. **Consistency and no interference.**

This is the cleanest possible setting for adjustment: the confounders are all
measured and we know it. Whatever goes wrong here cannot be blamed on unmeasured
confounding — which is what makes the results below awkward.

## Estimation

Start with replication 1, the way a single-dataset analysis would.

In [2]:
truth_first = first["true_ite"].mean()

single = pd.DataFrame(
    [
        ("naive difference in means", difference_in_means(first).estimate),
        ("g-computation", g_computation_ate(first, covariates=COVARIATES).estimate),
        ("IPW", ipw_ate(first, covariates=COVARIATES).estimate),
        ("AIPW", aipw_ate(first, covariates=COVARIATES).estimate),
        ("DML", double_machine_learning_ate(first, covariates=COVARIATES, n_splits=5, seed=0).estimate),
    ],
    columns=["estimator", "estimate"],
)
single["error"] = single["estimate"] - truth_first

print(f"replication 1, true ATE {truth_first:.3f}\n")
print(single.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

replication 1, true ATE 4.016

                estimator  estimate  error
naive difference in means     4.021  0.005
            g-computation     3.929 -0.087
                      IPW     3.460 -0.556
                     AIPW     3.970 -0.046
                      DML     3.913 -0.103


**Interpretation.** The naive estimate is 4.021 against a true 4.016 — an error
of 0.005, better than every adjusted estimator on this replication. IPW is the
worst at 0.56 off.

Taken alone this would be a strange thing to publish, and the temptation is to
call it a fluke of one draw. That is a testable claim, so let us test it.

Now the same panel across all ten replications. Each is a fresh draw of the
simulated outcomes from the same covariates, so this is a distribution of errors
rather than a single one.

In [3]:
records = []
for replication, subset in data.groupby("replication"):
    truth = subset["true_ite"].mean()
    estimates = {
        "naive": difference_in_means(subset).estimate,
        "g-computation": g_computation_ate(subset, covariates=COVARIATES).estimate,
        "IPW": ipw_ate(subset, covariates=COVARIATES).estimate,
        "AIPW": aipw_ate(subset, covariates=COVARIATES).estimate,
        "DML": double_machine_learning_ate(
            subset, covariates=COVARIATES, n_splits=5, seed=0
        ).estimate,
    }
    for name, value in estimates.items():
        records.append(
            {"replication": replication, "estimator": name, "error": value - truth}
        )

errors = pd.DataFrame(records)
summary = (
    errors.assign(abs_error=errors["error"].abs())
    .groupby("estimator")
    .agg(
        mean_error=("error", "mean"),
        mean_abs_error=("abs_error", "mean"),
        median_abs_error=("abs_error", "median"),
        worst=("abs_error", "max"),
    )
    .sort_values("mean_abs_error")
)
print(summary.to_string(float_format=lambda v: f"{v:.3f}"))

               mean_error  mean_abs_error  median_abs_error  worst
estimator                                                         
naive               0.192           0.262             0.111  1.633
AIPW               -0.322           0.378             0.180  2.112
g-computation      -0.720           0.742             0.156  5.703
DML                -0.767           0.772             0.212  5.708
IPW                -1.284           1.284             0.761  4.846


**Interpretation.** It was not a fluke. Across ten replications the naive
estimator has the lowest mean absolute error, 0.26, ahead of AIPW at 0.38 and
well ahead of IPW at 1.28. The adjusted estimators also share a negative mean
error — they systematically *undershoot* the true effect, while the naive
estimator is close to unbiased here.

The reason is not that adjustment is a bad idea. It is that adjustment is not
free. Every adjusted estimator fits nuisance models on 747 observations with 25
covariates and 139 treated units, using linear specifications that are not the
functional form the outcomes were simulated from. The variance and
misspecification that introduces exceeds the confounding bias it removes,
because on this benchmark the covariate imbalance — real as it is — happens to
translate into only a small bias in the mean.

Notebook 07 showed the opposite regime, where refusing to adjust costs 1.8 and
adjusting badly costs more than doing nothing. Both notebooks make the same
point from different sides: whether adjustment helps is an empirical question
about the specific data, and the output of an analysis never tells you the
answer.

In [4]:
per_replication = (
    errors.assign(abs_error=errors["error"].abs())
    .pivot(index="replication", columns="estimator", values="abs_error")
)
print("absolute error by replication:")
print(per_replication.to_string(float_format=lambda v: f"{v:.2f}"))

beats = int((per_replication["naive"] < per_replication["AIPW"]).sum())
print(f"\nreplications where naive beats AIPW: {beats} of {len(per_replication)}")
print(f"mean absolute error excluding replication 9:")
print(per_replication.drop(index=9).mean().sort_values().to_string(float_format=lambda v: f"{v:.3f}"))

absolute error by replication:
estimator    AIPW  DML  IPW  g-computation  naive
replication                                      
1            0.05 0.10 0.56           0.09   0.01
2            0.14 0.22 0.72           0.15   0.03
3            0.16 0.25 0.70           0.16   0.09
4            0.31 0.44 1.19           0.37   0.00
5            0.20 0.03 0.93           0.01   0.07
6            0.04 0.08 0.45           0.04   0.13
7            0.21 0.20 0.65           0.16   0.15
8            0.04 0.02 0.80           0.09   0.20
9            2.11 5.71 4.85           5.70   1.63
10           0.53 0.65 1.99           0.63   0.31

replications where naive beats AIPW: 8 of 10
mean absolute error excluding replication 9:
estimator
naive           0.110
AIPW            0.186
g-computation   0.190
DML             0.223
IPW             0.888


**Interpretation.** Naive wins in 8 of 10 replications, so the ranking is not an
artefact of averaging.

Replication 9 is the one that matters for how the summary reads. Every estimator
does badly there — g-computation and DML are off by 5.7 — and it is that single
replication that inflates their means. Drop it and the adjusted estimators
improve, but not equally: g-computation falls from 0.74 to 0.19 and DML from
0.77 to 0.22, while naive goes from 0.26 to 0.11. The adjusted estimators were
carrying most of the damage from the heavy-tailed replication, and excluding it
closes most of the gap — though naive still leads.

Neither table is wrong. But "mean absolute error over ten replications" and
"mean absolute error over the nine well-behaved replications" support different
conclusions, and a benchmark result is only as meaningful as the aggregation
choice behind it — a choice that is rarely reported.

## Diagnostics

Given that adjustment underperformed, the diagnostics should show us what the
adjusters were up against: how imbalanced the covariates are, and how well the
weighting fixes them.

In [5]:
before = balance_table(first, covariates=COVARIATES)
weights = ipw_weights(first, covariates=COVARIATES)
after = balance_table(first, covariates=COVARIATES, weights=weights)

print(f"worst |SMD| before weighting: {before['abs_smd'].max():.3f}")
print(f"worst |SMD| after weighting:  {after['abs_smd'].max():.3f}")
print(f"covariates above 0.1 before:  {int((before['abs_smd'] > 0.1).sum())} of {len(before)}")
print(f"covariates above 0.1 after:   {int((after['abs_smd'] > 0.1).sum())} of {len(after)}")

print(f"\nlargest IPW weight: {weights.max():.1f}")
effective = weights.sum() ** 2 / (weights**2).sum()
print(f"effective sample size: {effective:.0f} of {len(weights)}")
print(f"treated units: {int(first['treatment'].sum())}")

worst |SMD| before weighting: 0.395
worst |SMD| after weighting:  0.280
covariates above 0.1 before:  16 of 25
covariates above 0.1 after:   4 of 25

largest IPW weight: 25.7
effective sample size: 345 of 747
treated units: 139


**Interpretation.** The imbalance is real but moderate: worst standardized mean
difference 0.395 before weighting, with 16 of 25 covariates above the 0.1
threshold — against a worst of 1.67 in the Lalonde data. Weighting
brings it down to 0.280 and leaves 4 of 25 covariates above the 0.1 threshold —
better than Lalonde's 8 of 10, still not clean.

The overlap numbers explain IPW's poor showing. The largest weight is 25.7, and
the effective sample size is 345 of 747 — the estimator is working with less
than half the data it appears to have, and with only 139 treated units to begin
with. That is a thin basis for a reweighted estimate, and it shows up directly
as the worst mean absolute error in the panel.

IHDP is best known as a **CATE** benchmark, scored by PEHE. That is a harder
target than the ATE: it is graded per child, so errors cannot cancel.

In [6]:
def pehe(predicted: np.ndarray, true: np.ndarray) -> float:
    """Root mean squared error of individual effect estimates."""
    return float(np.sqrt(np.mean((np.asarray(predicted) - np.asarray(true)) ** 2)))


learner_scores = []
for replication, subset in data.groupby("replication"):
    features, true_effects = subset[COVARIATES], subset["true_ite"].to_numpy()
    learners = {
        "S-learner (linear)": SMetaLearner(),
        "T-learner (linear)": TMetaLearner(),
        "S-learner (boosted)": SMetaLearner(model=GradientBoostingRegressor(random_state=0)),
        "T-learner (boosted)": TMetaLearner(
            treated_model=GradientBoostingRegressor(random_state=0),
            control_model=GradientBoostingRegressor(random_state=0),
        ),
    }
    for name, learner in learners.items():
        learner.fit(subset, covariates=COVARIATES, treatment_col="treatment", outcome_col="outcome")
        predicted = learner.predict_cate(features)
        learner_scores.append(
            {
                "learner": name,
                "replication": replication,
                "pehe": pehe(predicted, true_effects),
                "cate_sd": float(np.std(predicted)),
                "true_sd": float(true_effects.std()),
            }
        )

scores = pd.DataFrame(learner_scores)
print("PEHE across replications (lower is better):")
print(
    scores.groupby("learner")
    .agg(mean=("pehe", "mean"), median=("pehe", "median"), worst=("pehe", "max"))
    .sort_values("median")
    .to_string(float_format=lambda v: f"{v:.3f}")
)
print("\nmean predicted CATE spread vs true spread:")
print(
    scores.groupby("learner")[["cate_sd", "true_sd"]]
    .mean()
    .to_string(float_format=lambda v: f"{v:.3f}")
)

PEHE across replications (lower is better):
                     mean  median  worst
learner                                 
S-learner (boosted) 2.208   0.606 13.289
T-learner (linear)  2.024   0.730 11.566
T-learner (boosted) 1.271   0.791  5.142
S-learner (linear)  4.673   1.236 27.949

mean predicted CATE spread vs true spread:
                     cate_sd  true_sd
learner                              
S-learner (boosted)    3.313    4.601
S-learner (linear)     0.000    4.601
T-learner (boosted)    4.601    4.601
T-learner (linear)     4.297    4.601


**Interpretation.** Two findings, one of them familiar.

**The linear S-learner reports a CATE standard deviation of 0.000 again.** It
did this on synthetic data in notebook 03, and it does it here on real
covariates: a linear model with treatment as an additive feature has no term in
which treatment interacts with anything, so its predicted effect is a constant
for every child. On a benchmark whose true effects have a spread of 4.6, it
reports no variation at all, and still produces a plausible-looking mean.

**Mean and median disagree about the winner.** By mean PEHE the boosted
T-learner leads; by median the boosted S-learner does, at 0.61 against 0.73. The
difference is replication 9 again, where the boosted S-learner scores 13.3. Both
statistics are computed from the same ten numbers, and which one you report
decides which method looks best.

## Uncertainty

Two sources here, and they are not the same size. Sampling variability within a
replication, and variation across replications.

In [7]:
interval = bootstrap_ate(
    first,
    estimator=aipw_ate,
    covariates=COVARIATES,
    n_bootstrap_samples=300,
    seed=0,
)

print(f"replication 1, AIPW: {interval.estimate:.3f}")
print(f"95% interval:        [{interval.lower:.3f}, {interval.upper:.3f}]")
print(f"true ATE:            {truth_first:.3f}")
print(f"covers truth:        {interval.lower <= truth_first <= interval.upper}")
print(f"interval width:      {interval.upper - interval.lower:.3f}")

spread = errors.loc[errors["estimator"] == "AIPW", "error"]
print(f"\nAIPW error across replications: {spread.min():.3f} to {spread.max():.3f}")
print(f"range: {spread.max() - spread.min():.3f}")

replication 1, AIPW: 3.970
95% interval:        [3.738, 4.163]
true ATE:            4.016
covers truth:        True
interval width:      0.425

AIPW error across replications: -2.112 to 0.199
range: 2.311


**Interpretation.** The within-replication interval is 0.43 wide and covers the
truth. The same estimator's error *across* replications spans 2.31 — more than
five times as much.

A confidence interval computed on one replication describes only the first of
those. It is a correct statement about sampling variability given this draw of
the simulated outcomes, and it says nothing about how the estimator behaves when
the outcome surface changes. Reporting a single-dataset interval as the
uncertainty of a *method* understates it fivefold here.

## Limitations

- **Ground truth exists only because the outcomes are simulated.** IHDP's
  covariates and treatment assignment are real, but its outcomes are not, so
  every score here is conditional on the simulation being a reasonable stand-in
  for a real response surface. It is a benchmark, not evidence about children.
- **"Naive wins" does not generalise, and is not advice.** It is a fact about
  this benchmark, these estimators, and default linear nuisance models. Notebook
  07 shows a regime where refusing to adjust is badly wrong. What generalises is
  that you cannot tell which regime you are in from the output.
- **Ten replications is few.** The literature typically uses 100 or 1000. With
  ten, one atypical replication moves every mean, as shown.
- **The replications are not exchangeable.** Replication 9 has a different
  outcome scale and effect size from the rest. Averaging across them mixes two
  problems.
- **Overlap is imperfect.** 139 treated units, a maximum weight of 25.7, and an
  effective sample size of 345 of 747. Weighted estimators are working with far
  less information than the row count suggests.
- **Nuisance models are untuned.** Linear by default, boosted where stated, with
  no cross-validation. A tuned comparison could reorder the panel — which is
  itself a caution about benchmark rankings.
- **PEHE is not decision-relevant on its own.** A model can rank children
  correctly for targeting while scoring poorly on PEHE, and notebook 11 is where
  that distinction is made.